**Programmer: python_scripts (Abhijith Warrier)**

**PYTHON SCRIPT TO _DEMONSTRATE DOCUMENT CHUNKING STRATEGIES_ 🐍**

Compare fixed-size chunks, overlap, and Markdown section chunks before embedding and retrieval. This is a retrieval lab, not a full answer-generating RAG application.

Requirements: `python -m pip install sentence-transformers faiss-cpu numpy`. The first run downloads `all-MiniLM-L6-v2`; subsequent runs reuse the cache. No paid API or API key is required.

---
## 📚 1. Load the Model and Fictional Documents

These short policies are illustrative, not real customer policies. We retain the originals so chunk offsets and source text can be checked.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import hashlib
import json
import re
from pathlib import Path
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2", cache_folder=".model_cache", device="cpu"
)
print("Embedding dimensions:", model.get_sentence_embedding_dimension())
print("Maximum input tokens:", model.max_seq_length)

documents = [
    {"id": "support", "title": "Customer Support Policy", "file": "support_policy.md", "text": """# Customer Support Policy

## Refunds
Customers can request a refund within 30 days of purchase. The order ID is required for every refund request. Refunds are returned to the original payment method.

## Processing
Approved refunds are processed within 5 business days. Banks may take additional time to display the credit.

## Exclusions
Gift cards are non-refundable. Downloaded digital products are not eligible for a refund after access has started.

## Contact
Send refund questions to the support team through the account portal. Include your order ID and a description of the issue.
"""},
    {"id": "account", "title": "Account Guide", "file": "account_guide.md", "text": """# Account Guide

## Passwords
Password reset links expire after 15 minutes. Request a new link from the sign-in page if the old link has expired.

## Security
Two-factor authentication can be enabled in account settings. Save recovery codes in a secure location before changing your phone.

## Email
To change the account email address, confirm the new address using the verification message. Existing subscriptions remain active.
"""},
    {"id": "shipping", "title": "Shipping Guide", "file": "shipping_guide.md", "text": """# Shipping Guide

## Tracking
Tracking links become available after the parcel is dispatched. Find the link in the shipping confirmation email or the orders page.

## Delivery
Standard delivery usually takes 3 to 7 business days after dispatch. Delivery estimates depend on the destination and carrier.

## Address changes
Shipping addresses can be changed before dispatch. Contact support immediately if an order contains an incorrect address.
"""},
]
# These are fictional policies, created only for this tutorial.
for document in documents:
    Path(document["file"]).write_text(document["text"], encoding="utf-8")

Embedding dimensions: 384
Maximum input tokens: 256


---
## ✂️ 2. Define Three Strategies

- **Fixed:** 120-character windows with no overlap. Simple, but may split a sentence.
- **Overlap:** the same 120-character windows with 40 characters repeated. This can retain boundary context while increasing indexed text.
- **Section-aware:** respect `##` headings, then use a 240-character fallback for an oversized section. This example's sections fit; the fallback can split sentences.

Characters keep this demonstration inspectable; they are not tokens. Production chunking should consider model token limits, document structure, tables, and the questions being asked.

Each chunk stores its source, section, content-version hash, and `[start, end)` character offsets.

In [2]:
def make_chunk(document, start, end):
    text = document["text"]
    while start < end and text[start].isspace():
        start += 1
    while end > start and text[end - 1].isspace():
        end -= 1
    headings = list(re.finditer(r"(?m)^## (.+)$", text[:start]))
    section = headings[-1].group(1) if headings else "Introduction"
    version = hashlib.sha256(text.encode()).hexdigest()[:8]
    return {"chunk_id": f"{document['id']}:{version}:{start}-{end}",
            "doc_id": document["id"], "title": document["title"],
            "source": document["file"], "section": section,
            "start": start, "end": end, "text": text[start:end]}


def fixed_chunks(document, size=120, overlap=0):
    if not 0 <= overlap < size:
        raise ValueError("Require 0 <= overlap < size")
    chunks = []
    for start in range(0, len(document["text"]), size - overlap):
        end = min(start + size, len(document["text"]))
        chunk = make_chunk(document, start, end)
        if chunk["text"]:
            chunks.append(chunk)
        if end == len(document["text"]):
            break
    return chunks


def section_chunks(document, max_chars=240):
    # This small demo recognises Markdown level-two headings.
    text = document["text"]
    headings = list(re.finditer(r"(?m)^## (.+)$", text))
    starts = [0] + [match.end() for match in headings]
    ends = [match.start() for match in headings] + [len(text)]
    chunks = []
    for start, end in zip(starts, ends):
        for position in range(start, end, max_chars):
            chunk = make_chunk(document, position, min(position + max_chars, end))
            if chunk["text"]:
                chunks.append(chunk)
    return chunks

strategies = {
    "Fixed": [chunk for doc in documents for chunk in fixed_chunks(doc)],
    "Overlap": [chunk for doc in documents for chunk in fixed_chunks(doc, overlap=40)],
    "Section": [chunk for doc in documents for chunk in section_chunks(doc)],
}
originals = {doc["id"]: doc["text"] for doc in documents}
for name, chunks in strategies.items():
    assert all(c["text"] == originals[c["doc_id"]][c["start"]:c["end"]] for c in chunks)
    print(name, "chunks:", len(chunks), "indexed characters:", sum(len(c["text"]) for c in chunks))

Fixed chunks: 13 indexed characters: 1461
Overlap chunks: 18 indexed characters: 2061
Section chunks: 13 indexed characters: 1317


---
## 🔍 3. Embed and Retrieve

Use the same embedding model for chunks and questions. Normalised vectors plus `IndexFlatIP` give cosine similarity. Check token lengths before embedding to avoid silent truncation. Similarity is a ranking signal, not proof that a chunk answers the question.

[Sentence Transformers input limits](https://sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html)

In [3]:
def build_index(chunks):
    texts = [chunk["text"] for chunk in chunks]
    lengths = [len(model.tokenizer.encode(text, add_special_tokens=True)) for text in texts]
    assert max(lengths) <= model.max_seq_length, "A chunk would be truncated"
    vectors = model.encode(texts, normalize_embeddings=True, convert_to_numpy=True)
    vectors = np.asarray(vectors, dtype="float32")
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    return index


def retrieve(question, chunks, index, top_k=2):
    if top_k < 1:
        raise ValueError("top_k must be positive")
    if not chunks:
        return []
    vector = model.encode([question], normalize_embeddings=True, convert_to_numpy=True)
    scores, ids = index.search(np.asarray(vector, dtype="float32"), min(top_k, len(chunks)))
    return [{**chunks[i], "score": float(score)} for i, score in zip(ids[0], scores[0]) if i >= 0]

indexes = {name: build_index(chunks) for name, chunks in strategies.items()}
question = "How long does an approved refund take to process?"
for name, chunks in strategies.items():
    print("\n", name)
    for hit in retrieve(question, chunks, indexes[name]):
        print(hit["chunk_id"], round(hit["score"], 3), repr(hit["text"]))


 Fixed
support:f346ddf6:0-120 0.555 '# Customer Support Policy\n\n## Refunds\nCustomers can request a refund within 30 days of purchase. The order ID is require'
support:f346ddf6:120-240 0.433 'd for every refund request. Refunds are returned to the original payment method.\n\n## Processing\nApproved refunds are pro'

 Overlap
support:f346ddf6:160-280 0.746 'returned to the original payment method.\n\n## Processing\nApproved refunds are processed within 5 business days. Banks may'
support:f346ddf6:0-120 0.555 '# Customer Support Policy\n\n## Refunds\nCustomers can request a refund within 30 days of purchase. The order ID is require'

 Section
support:f346ddf6:216-324 0.781 'Approved refunds are processed within 5 business days. Banks may take additional time to display the credit.'
support:f346ddf6:38-200 0.605 'Customers can request a refund within 30 days of purchase. The order ID is required for every refund request. Refunds are returned to the original payment method.'


---
## ✅ 4. Check Evidence Coverage

Use four hand-labelled questions. A hit requires a retrieved chunk from the expected document containing the **complete exact evidence sentence**. Also count whether that sentence survives anywhere in the chunk set.

This is a strict, small diagnostic—not a general relevance benchmark or proof that one strategy is best. Strategies use different chunk lengths and may return different amounts of context.

In [4]:
tests = [
    ("What is the deadline for requesting a refund?", "support", "Customers can request a refund within 30 days of purchase."),
    ("How long does an approved refund take to process?", "support", "Approved refunds are processed within 5 business days."),
    ("When does a password reset link expire?", "account", "Password reset links expire after 15 minutes."),
    ("When can I get a parcel tracking link?", "shipping", "Tracking links become available after the parcel is dispatched."),
]
results = []
for name, chunks in strategies.items():
    preserved = hits = 0
    for query, doc_id, evidence in tests:
        relevant = lambda chunk: chunk["doc_id"] == doc_id and evidence in chunk["text"]
        preserved += any(relevant(chunk) for chunk in chunks)
        hits += any(relevant(chunk) for chunk in retrieve(query, chunks, indexes[name]))
    row = {"strategy": name, "chunks": len(chunks),
           "indexed_characters": sum(len(c["text"]) for c in chunks),
           "evidence_preserved": preserved, "hits_at_2": hits, "questions": len(tests)}
    results.append(row)
    print(row)
Path("chunking_results.json").write_text(json.dumps(results, indent=2))
Path("chunking_chunks.json").write_text(json.dumps(strategies, indent=2))

{'strategy': 'Fixed', 'chunks': 13, 'indexed_characters': 1461, 'evidence_preserved': 3, 'hits_at_2': 3, 'questions': 4}
{'strategy': 'Overlap', 'chunks': 18, 'indexed_characters': 2061, 'evidence_preserved': 4, 'hits_at_2': 4, 'questions': 4}
{'strategy': 'Section', 'chunks': 13, 'indexed_characters': 1317, 'evidence_preserved': 4, 'hits_at_2': 4, 'questions': 4}


15687

---
## 📌 Takeaways

Chunking determines which context retrieval can return. Overlap repeats text rather than adding new information; structure can keep a topic together but still needs size limits. Keep source metadata attached, inspect missed evidence, and tune on representative questions.

Do not infer a universal chunk size or ranking from four short questions. The next lab uses section chunks to demonstrate traceable source references.


---